# One-model joint planner+executor (single Qwen, LoRA)

One Qwen emits `instruction -> plan tokens (PLAN head) -> EOP -> response (LANGUAGE head)`.
Same backbone, two heads, phase switch. Teacher-forced on the 800-op plan corpus with
`L = CE_plan + lambda*CE_response`. Trainable: planner LoRA + plan head + plan embeddings;
Qwen's language head/embeddings stay frozen. Shows **train and held-out loss falling**.

**Runtime -> Change runtime type -> T4 GPU**, then Run all.

In [ ]:
!git clone -b one-model-joint https://github.com/sinha-k-prat/latent_agentic_planning.git
%cd latent_agentic_planning
!pip -q install -r requirements.txt
!pip -q uninstall -y torchao  # Colab torchao 0.10 clashes with newer peft

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Full corpus: 700 train / 100 held / 64 calib, on GPU. ~10-20 min on a T4.
# lam_kl = response KL anchor (preserves fluent answering); early-stop on held planned loss.
!python examples/train_joint.py --train 700 --held 100 --calib 64 --epochs 6 \
  --max_resp 64 --accum 8 --lr 1e-4 --lam_kl 0.3 --kl_batch 8 --patience 3 \
  --device cuda --base Qwen/Qwen2.5-0.5B-Instruct

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv('examples/joint_metrics.csv')
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(df.epoch, df.train_total, marker='.', label='train total')
ax[0].plot(df.epoch, df.held_plan, marker='.', label='held plan CE')
ax[0].plot(df.epoch, df.held_resp_planned, marker='.', label='held response CE (planned)')
ax[0].legend(); ax[0].set_title('losses falling'); ax[0].set_xlabel('epoch'); ax[0].grid(alpha=.3)
# THE key plot: does the plan help? with-plan response NLL should sit BELOW no-plan.
ax[1].plot(df.epoch, df.held_resp_unplanned, marker='.', label='response NLL: NO plan')
ax[1].plot(df.epoch, df.held_resp_planned, marker='.', label='response NLL: WITH plan')
ax[1].legend(); ax[1].set_title('plan ablation (with-plan below = plan helps)')
ax[1].set_xlabel('epoch'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(df[['epoch', 'held_resp_planned', 'held_resp_unplanned',
          'gap_unplanned_minus_planned']].to_string(index=False))